In [8]:
from brollm import BaseContract
from broflow import BaseTask, TaskRegistry, Flow
from broskill import SkillControl, ToolControl, Skill, Tool, Arg
from broskill.processing.tool import to_args

from pathlib import Path
import subprocess
import sys
from dataclasses import dataclass
from enum import StrEnum
from typing import Any

ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
tc = ToolControl(sc)

sc.list_skills()

[Skill(name='read-file', description="Read the contents of a specific file, or list files matching a pattern. Use when the user wants to see what's in a file, or wants to find files matching a pattern.", version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/read-file'), tags=['filesystem'], keywords=None, default=False, status='experiment'),
 Skill(name='tell-joke', description='Tell a joke on request -- dad jokes, puns, or a mix of both. Use when the user asks for a joke, wants to be entertained, or needs a laugh.', version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/tell-joke'), tags=['fun', 'entertainment'], keywords=None, default=False, status='experiment')]

In [9]:
def register_tool(tool) -> dict:
    """broskill Tool -> Bedrock toolSpec (different shape from OpenAI/Ollama's
    {"type": "function", "function": {...}} -- Bedrock nests the JSON schema
    one level deeper, under inputSchema.json)."""
    return {
        "toolSpec": {
            "name": tool.name,
            "description": tool.description,
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        arg.name: {"type": arg.type, "description": arg.description}
                        for arg in tool.args
                    },
                    "required": [arg.name for arg in tool.args if arg.required],
                }
            },
        }
    }

In [10]:
import boto3
from brollm import BaseContract
from typing import Any

REGION = "us-east-1"
# MODEL_ID = "qwen.qwen3-vl-235b-a22b"
# MODEL_ID = "google.gemma-3-27b-it"
MODEL_ID = "openai.gpt-oss-20b-1:0"

client = boto3.client("bedrock-runtime", region_name=REGION)

def input_fn(
    system_prompt: str,
    messages: list[dict],
    tool_registry: list[dict] | None = None,
) -> dict:
    kwargs = {
        "modelId": MODEL_ID,
        "system": [{"text": system_prompt}],
        "messages": messages,
    }
    if tool_registry:
        kwargs["toolConfig"] = {"tools": tool_registry}
    return client.converse(**kwargs)

def output_fn(response: dict) -> Any:
    return response

llm = BaseContract(input_fn=input_fn, output_fn=output_fn)

In [11]:
def UserMessage(text: str) -> dict[str, Any]:
    return {"role": "user", "content": [{"text": text}]}

def AssistantMessage(response: dict) -> dict:
    # identity, kept for parity with dev.ipynb -- Bedrock's
    # response["output"]["message"] is already {"role": "assistant", "content": [...]},
    # so there's nothing to reshape here, unlike Ollama's response['message'].
    return response["output"]["message"]

def ToolResultBlock(tool_use_id: str, content) -> dict:
    # a single block, not a message -- see the loop below for why these get
    # batched into ONE user message instead of one message per tool call.
    return {"toolResult": {"toolUseId": tool_use_id, "content": [{"text": str(content)}]}}

In [12]:
TOOL_REGISTRYS = [
    {
        "toolSpec": {
            "name": "load_skill",
            "description": (
                "Load the full instructions for a registered skill by name. "
                "Call this only when the current task clearly matches that "
                "skill's description."
            ),
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {"skill_name": {"type": "string"}},
                    "required": ["skill_name"],
                }
            },
        }
    },
    {
        "toolSpec": {
            "name": "load_skill_extension",
            "description": (
                "Load the full contents of one reference/asset document belonging to "
                "an already-loaded skill. Call this only when that skill's "
                "instructions point you to a specific reference/asset file for more "
                "detail — don't call it speculatively."
            ),
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "skill_name": {
                            "type": "string",
                            "description": "The name of the skill whose reference/asset you want to load."
                        },
                        "path": {
                            "type": "string",
                            "description": "The reference/asset file's path exactly as shown in the skill's instructions, e.g. 'references/aws.md' or 'assets/color.ts'."
                        }
                    },
                    "required": ["skill_name", "path"],
                }
            },
        }
    },
    {
        "toolSpec": {
            "name": "ask_followup_question",
            "description": (
                "Signal that you need the user to answer something before you can "
                "continue. Write the actual question as your normal response "
                "content, then call this tool with no arguments to pause the turn "
                "and wait for their reply."
            ),
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {},
                    "required": [],
                }
            },
        }
    },
]

In [13]:
TOOLS = {
    "load_skill_extension": sc.load_skill_extension,
    "load_skill": sc.load_skill,
    "ask_followup_question": None,
}

In [14]:
available_skills = "\n".join(f"- {s.name}: {s.description}" for s in sc.list_skills())

# system_prompt = (
#     "You're a helpful assistant.\n\n"
#     f"Available skills:\n{available_skills}"
# )

system_prompt = (
    "You have access to registered skills below. If the user's request matches "
    "one of them, you MUST call load_skill to get its full instructions BEFORE "
    "responding -- never answer from your own general knowledge when a matching "
    "skill exists.\n\n"
    f"Available skills:\n{available_skills}"
)


content = "I'm so bored. Can you tell me a joke?"
messages = [UserMessage(content)]

In [15]:
# response = llm(system_prompt, messages, tool_registry=TOOL_REGISTRYS)
# response

In [16]:
while True:
    response = llm(system_prompt, messages, tool_registry=TOOL_REGISTRYS)
    message = AssistantMessage(response)
    messages.append(message)
    print(message)

    if response["stopReason"] != "tool_use":
        break

    text_blocks = [b["text"] for b in message["content"] if "text" in b]
    _content = "\n".join(text_blocks)

    # Bedrock can return several toolUse blocks in ONE turn (parallel tool
    # calls) -- collect every result here, then send them back as a single
    # user message, never one message per tool call. Two separate user
    # messages in a row would violate Bedrock's strict role alternation.
    tool_result_blocks = []
    for block in message["content"]:
        if "toolUse" not in block:
            continue
        tu = block["toolUse"]
        tool_use_id, tool_name, tool_args = tu["toolUseId"], tu["name"], tu.get("input") or {}

        if tool_name == "ask_followup_question":
            if not _content:
                result = "It looks like you want to ask me something?"
            else:
                result = input(_content)
        elif tool_name in TOOLS:
            result = TOOLS[tool_name](**tool_args)
        else:
            result = f"Tool {tool_name} not found in registry."

        tool_result_blocks.append(ToolResultBlock(tool_use_id, result))

    messages.append({"role": "user", "content": tool_result_blocks})

{'role': 'assistant', 'content': [{'reasoningContent': {'reasoningText': {'text': 'User wants a joke. We have a skill "tell-joke". Must load skill.'}}}, {'toolUse': {'toolUseId': 'tooluse_bxvJbvUmmo5tfSesswqT2y', 'name': 'load_skill', 'input': {'skill_name': 'tell-joke'}}}]}
{'role': 'assistant', 'content': [{'reasoningContent': {'reasoningText': {'text': "We need to follow the skill: Ask which kind of joke they'd like. The user didn't specify. So we should ask a short clarifying question. Then call ask_followup_question."}}}, {'toolUse': {'toolUseId': 'tooluse_RVA9FtiTcPPvkNM581yRvZ', 'name': 'ask_followup_question', 'input': {}}}]}
{'role': 'assistant', 'content': [{'reasoningContent': {'reasoningText': {'text': "We need to respond with a clarifying question, then call ask_followup_question. According to skill, we should ask which kind of joke they'd like: dad jokes, puns, or a mix. Then call ask_followup_question. Let's produce that."}}}, {'text': 'Sure thing! What kind of joke are 

In [17]:
messages

[{'role': 'user',
  'content': [{'text': "I'm so bored. Can you tell me a joke?"}]},
 {'role': 'assistant',
  'content': [{'reasoningContent': {'reasoningText': {'text': 'User wants a joke. We have a skill "tell-joke". Must load skill.'}}},
   {'toolUse': {'toolUseId': 'tooluse_bxvJbvUmmo5tfSesswqT2y',
     'name': 'load_skill',
     'input': {'skill_name': 'tell-joke'}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_bxvJbvUmmo5tfSesswqT2y',
     'content': [{'text': "# Tell Joke\n\n## Instructions\n\n- Ask the user which kind of joke they'd like: dad jokes, puns, or a mix of both.\n- If they already said which kind in their request, don't ask again -- just tell one.\n- If their answer is unclear, ask a short clarifying question directly in your reply. Call `ask_followup_question`.\n- Tell exactly one joke at a time, in your own words. Don't paste a reference file's\n  contents back verbatim.\n- Keep it short, and don't explain the joke afterward -- a joke th

In [18]:
response

{'ResponseMetadata': {'RequestId': '575f32ff-e62d-4054-9e09-1003774de0b2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 13 Sep 2026 18:20:28 GMT',
   'content-type': 'application/json',
   'content-length': '570',
   'connection': 'keep-alive',
   'x-amzn-requestid': '575f32ff-e62d-4054-9e09-1003774de0b2'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'reasoningContent': {'reasoningText': {'text': "We need to respond with a clarifying question, then call ask_followup_question. According to skill, we should ask which kind of joke they'd like: dad jokes, puns, or a mix. Then call ask_followup_question. Let's produce that."}}},
    {'text': 'Sure thing! What kind of joke are you in the mood for—dad jokes, puns, or a mix of both?'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 762, 'outputTokens': 88, 'totalTokens': 850},
 'metrics': {'latencyMs': 954}}